In [1]:
import re
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from time import sleep

In [2]:
ruta = r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\df_tilos_limpio.parquet"
df = pd.read_parquet(ruta)

In [3]:
# ===============================
# 3. Sacar muestra aleatoria de 10 licitaciones
# ===============================

muestra = df.sample(
    n=10,
    random_state=42
).copy()

muestra.shape

(10, 28)

In [6]:
muestra.to_parquet(
    "C:/Users/Usuario/TFM/TFM_scraping_contratacion_estado/data/silver/muestra_10.parquet",
    index=False
)

In [7]:
# ===============================
# 5. Función para validar URLs
# ===============================

def validar_url(url, timeout=15):
    """
    Valida si una URL responde correctamente.
    """

    if pd.isna(url):
        return None, False, "URL vacía"

    try:
        respuesta = requests.get(url, timeout=timeout)

        codigo_estado = respuesta.status_code
        url_accesible = respuesta.status_code == 200

        return codigo_estado, url_accesible, None

    except Exception as error:
        return None, False, str(error)

In [8]:
# ===============================
# 6. Validar URLs de la muestra
# ===============================

resultados_url = []

for _, fila in muestra.iterrows():
    codigo_estado, url_accesible, error_acceso = validar_url(
        fila["detail_url"]
    )

    resultados_url.append(
        {
            "licitacion_id": fila["licitacion_id"],
            "codigo_estado_detail_url": codigo_estado,
            "url_accesible": url_accesible,
            "error_acceso": error_acceso
        }
    )

    sleep(1)

validacion_urls = pd.DataFrame(resultados_url)

validacion_urls

,licitacion_id,codigo_estado_detail_url,url_accesible,error_acceso
0,3b417cfd32be2e0f,200,True,None
1,c859b544e1b65ae1,200,True,None
2,674815cdf25e757d,200,True,None
3,d54242588e76eb26,200,True,None
4,e0aadecc068e8f46,200,True,None
5,710a5a77b1e2940a,200,True,None
6,38fff7ade079f8ea,200,True,None
7,a3bab454da5a382a,200,True,None
8,925a9e399f5a9a67,200,True,None
9,0f9fcaf0ce98be3d,200,True,None


In [9]:
# ===============================
# 7. Unir validación a la muestra
# ===============================

muestra = muestra.merge(
    validacion_urls,
    on="licitacion_id",
    how="left"
)

muestra[
    [
        "licitacion_id",
        "detail_url",
        "codigo_estado_detail_url",
        "url_accesible",
        "error_acceso"
    ]
]

,licitacion_id,detail_url,codigo_estado_detail_url,url_accesible,error_acceso
0,3b417cfd32be2e0f,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
1,c859b544e1b65ae1,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
2,674815cdf25e757d,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
3,d54242588e76eb26,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
4,e0aadecc068e8f46,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
5,710a5a77b1e2940a,https://contratos-publicos.comunidad.madrid/co...,200,True,None
6,38fff7ade079f8ea,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
7,a3bab454da5a382a,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
8,925a9e399f5a9a67,https://www.contratosdegalicia.gal/licitacion?...,200,True,None
9,0f9fcaf0ce98be3d,https://www.contratosdegalicia.gal/licitacion?...,200,True,None


In [10]:
muestra["url_accesible"].value_counts(dropna=False)

url_accesible
True    10
Name: count, dtype: int64

In [11]:
# ===============================
# 8. Detectar portal
# ===============================

def detectar_portal(detail_url):
    """
    Identifica el portal de contratación a partir de la URL.
    """

    if pd.isna(detail_url):
        return "sin_url"

    if "contratosdegalicia.gal" in detail_url:
        return "galicia"

    if "contratos-publicos.comunidad.madrid" in detail_url:
        return "madrid"

    return "otro"

In [12]:
muestra["portal"] = muestra["detail_url"].apply(detectar_portal)

muestra[
    [
        "licitacion_id",
        "detail_url",
        "portal"
    ]
]

,licitacion_id,detail_url,portal
0,3b417cfd32be2e0f,https://www.contratosdegalicia.gal/licitacion?...,galicia
1,c859b544e1b65ae1,https://www.contratosdegalicia.gal/licitacion?...,galicia
2,674815cdf25e757d,https://www.contratosdegalicia.gal/licitacion?...,galicia
3,d54242588e76eb26,https://www.contratosdegalicia.gal/licitacion?...,galicia
4,e0aadecc068e8f46,https://www.contratosdegalicia.gal/licitacion?...,galicia
5,710a5a77b1e2940a,https://contratos-publicos.comunidad.madrid/co...,madrid
6,38fff7ade079f8ea,https://www.contratosdegalicia.gal/licitacion?...,galicia
7,a3bab454da5a382a,https://www.contratosdegalicia.gal/licitacion?...,galicia
8,925a9e399f5a9a67,https://www.contratosdegalicia.gal/licitacion?...,galicia
9,0f9fcaf0ce98be3d,https://www.contratosdegalicia.gal/licitacion?...,galicia


In [13]:
muestra["portal"].value_counts()

portal
galicia    9
madrid     1
Name: count, dtype: int64

In [14]:
# ===============================
# 9. Seleccionar licitación Galicia
# ===============================

muestra_galicia = muestra[muestra["portal"] == "galicia"].copy()

licitacion_prueba_galicia = muestra_galicia.iloc[0]

licitacion_id_galicia = licitacion_prueba_galicia["licitacion_id"]
url_galicia = licitacion_prueba_galicia["detail_url"]

print("Licitación:", licitacion_id_galicia)
print("URL:", url_galicia)

Licitación: 3b417cfd32be2e0f
URL: https://www.contratosdegalicia.gal/licitacion?N=817889


In [15]:
# ===============================
# 10. Extraer texto visible Galicia
# ===============================

respuesta = requests.get(url_galicia, timeout=20)
respuesta.raise_for_status()

soup_galicia = BeautifulSoup(respuesta.text, "html.parser")

texto_galicia = soup_galicia.get_text("\n", strip=True)

lineas_galicia = texto_galicia.split("\n")

df_lineas_galicia = pd.DataFrame({
    "orden": range(len(lineas_galicia)),
    "linea": lineas_galicia
})

df_lineas_galicia.head(100)

,orden,linea
0,0,Detalle procedemento: 817889 - Contratos Públi...
1,1,Galego
2,2,|
3,3,Castellano
4,4,Presentación
...,...,...
95,95,0
96,96,Máximo
97,97,Non
98,98,Prazo execución acordo marco


In [16]:
# ===============================
# 11. Funciones auxiliares
# ===============================

def valor_siguiente(df_lineas, etiqueta):
    """
    Busca una etiqueta y devuelve la línea inmediatamente posterior.
    """

    coincidencias = df_lineas[
        df_lineas["linea"].str.contains(
            etiqueta,
            case=False,
            na=False,
            regex=False
        )
    ]

    if coincidencias.empty:
        return None

    orden = coincidencias.iloc[0]["orden"]

    valor = df_lineas.loc[
        df_lineas["orden"] == orden + 1,
        "linea"
    ]

    if valor.empty:
        return None

    return valor.iloc[0]


def valor_con_salto(df_lineas, etiqueta, salto):
    """
    Busca una etiqueta y devuelve la línea ubicada n posiciones después.
    """

    coincidencias = df_lineas[
        df_lineas["linea"].str.contains(
            etiqueta,
            case=False,
            na=False,
            regex=False
        )
    ]

    if coincidencias.empty:
        return None

    orden = coincidencias.iloc[0]["orden"] + salto

    valor = df_lineas.loc[
        df_lineas["orden"] == orden,
        "linea"
    ]

    if valor.empty:
        return None

    return valor.iloc[0]

In [17]:
# ===============================
# 12. Extraer cabecera Galicia
# ===============================

info_galicia = {
    "licitacion_id": licitacion_id_galicia,
    "detail_url": url_galicia,

    "estado_procedimiento_web": valor_siguiente(
        df_lineas_galicia,
        "Estado do procedemento"
    ),

    "organo_contratacion_web": valor_con_salto(
        df_lineas_galicia,
        "Estado do procedemento",
        salto=2
    ),

    "objeto_web": valor_siguiente(
        df_lineas_galicia,
        "Obxecto"
    ),

    "tipo_tramitacion_web": valor_siguiente(
        df_lineas_galicia,
        "Tipo de tramitación"
    ),

    "tipo_procedimiento_web": valor_siguiente(
        df_lineas_galicia,
        "Tipo de procedemento"
    ),

    "tipo_contrato_web": valor_siguiente(
        df_lineas_galicia,
        "Tipo de contrato"
    ),

    "presupuesto_base_web": valor_siguiente(
        df_lineas_galicia,
        "Orzamento base de licitación"
    ),

    "num_lotes_web": valor_siguiente(
        df_lineas_galicia,
        "Nº lotes"
    ),

    "valor_estimado_web": valor_siguiente(
        df_lineas_galicia,
        "Valor estimado"
    ),

    "tipo_financiamiento_web": valor_siguiente(
        df_lineas_galicia,
        "Tipo de financiamento"
    )
}

info_galicia = pd.DataFrame([info_galicia])

info_galicia.T

,0
licitacion_id,3b417cfd32be2e0f
detail_url,https://www.contratosdegalicia.gal/licitacion?...
estado_procedimiento_web,Adxudicado
organo_contratacion_web,Consellería de Sanidade - SERGAS
objeto_web,PSM001937990 708506 biombo pregable 7 corpos...
tipo_tramitacion_web,De Emerxencia
tipo_procedimiento_web,Procedemento de emerxencia
tipo_contrato_web,Subministracións
presupuesto_base_web,"7.170,00"
num_lotes_web,_


In [18]:
# ===============================
# 13. Función para extraer resolución Galicia
# ===============================

def extraer_resolucion_galicia(df_lineas):
    """
    Extrae la primera fila de la tabla de resolución en Galicia.
    """

    coincidencias = df_lineas[
        df_lineas["linea"].str.contains(
            "Datos da resolución do procedemento",
            case=False,
            na=False,
            regex=False
        )
    ]

    if coincidencias.empty:
        return {
            "lote_resolucion_web": None,
            "participacion_resolucion_web": None,
            "resolucion_web": None,
            "adjudicatario_web": None,
            "importe_adjudicado_web": None,
            "fecha_difusion_resolucion_web": None,
            "plazo_ejecucion_resolucion_web": None
        }

    inicio = coincidencias.iloc[0]["orden"]

    return {
        "lote_resolucion_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 9,
            "linea"
        ].iloc[0],

        "participacion_resolucion_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 10,
            "linea"
        ].iloc[0],

        "resolucion_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 11,
            "linea"
        ].iloc[0],

        "adjudicatario_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 12,
            "linea"
        ].iloc[0],

        "importe_adjudicado_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 13,
            "linea"
        ].iloc[0],

        "fecha_difusion_resolucion_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 14,
            "linea"
        ].iloc[0],

        "plazo_ejecucion_resolucion_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 15,
            "linea"
        ].iloc[0]
    }

In [19]:
# ===============================
# 14. Aplicar resolución Galicia
# ===============================

datos_resolucion_galicia = extraer_resolucion_galicia(df_lineas_galicia)

for columna, valor in datos_resolucion_galicia.items():
    info_galicia[columna] = valor

info_galicia.T

,0
licitacion_id,3b417cfd32be2e0f
detail_url,https://www.contratosdegalicia.gal/licitacion?...
estado_procedimiento_web,Adxudicado
organo_contratacion_web,Consellería de Sanidade - SERGAS
objeto_web,PSM001937990 708506 biombo pregable 7 corpos...
tipo_tramitacion_web,De Emerxencia
tipo_procedimiento_web,Procedemento de emerxencia
tipo_contrato_web,Subministracións
presupuesto_base_web,"7.170,00"
num_lotes_web,_


In [20]:
# ===============================
# 15. Función órgano competente recurso Galicia
# ===============================

def extraer_organo_recurso_galicia(df_lineas):
    """
    Extrae los datos del bloque:
    Órgano competente para resolver o recurso.
    """

    coincidencias = df_lineas[
        df_lineas["linea"].str.contains(
            "Órgano competente para resolver o recurso",
            case=False,
            na=False,
            regex=False
        )
    ]

    if coincidencias.empty:
        return {
            "organo_recurso_web": None,
            "direccion_recurso_web": None,
            "localidad_recurso_web": None,
            "cp_recurso_web": None,
            "telefono_recurso_web": None,
            "fax_recurso_web": None,
            "correo_recurso_web": None
        }

    inicio = coincidencias.iloc[0]["orden"]

    return {
        "organo_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 2,
            "linea"
        ].iloc[0],

        "direccion_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 4,
            "linea"
        ].iloc[0],

        "localidad_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 6,
            "linea"
        ].iloc[0],

        "cp_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 8,
            "linea"
        ].iloc[0],

        "telefono_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 10,
            "linea"
        ].iloc[0],

        "fax_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 12,
            "linea"
        ].iloc[0],

        "correo_recurso_web": df_lineas.loc[
            df_lineas["orden"] == inicio + 14,
            "linea"
        ].iloc[0]
    }

In [21]:
# ===============================
# 16. Aplicar órgano recurso Galicia
# ===============================

datos_recurso_galicia = extraer_organo_recurso_galicia(df_lineas_galicia)

for columna, valor in datos_recurso_galicia.items():
    info_galicia[columna] = valor

info_galicia.T

,0
licitacion_id,3b417cfd32be2e0f
detail_url,https://www.contratosdegalicia.gal/licitacion?...
estado_procedimiento_web,Adxudicado
organo_contratacion_web,Consellería de Sanidade - SERGAS
objeto_web,PSM001937990 708506 biombo pregable 7 corpos...
tipo_tramitacion_web,De Emerxencia
tipo_procedimiento_web,Procedemento de emerxencia
tipo_contrato_web,Subministracións
presupuesto_base_web,"7.170,00"
num_lotes_web,_


In [22]:
# ===============================
# 17. Función publicación, CPV y NUT Galicia
# ===============================

def extraer_publicacion_cpv_nut_galicia(df_lineas):
    """
    Extrae datos de medios de publicación, CPV y NUT.
    """

    datos = {
        "fecha_difusion_plataforma_web": None,
        "sello_web": None,
        "cpv_web": None,
        "cpv_lote_web": None,
        "cpv_fecha_difusion_web": None,
        "nut_web": None,
        "nut_lote_web": None,
        "nut_fecha_difusion_web": None
    }

    # Sello
    coincidencias_sello = df_lineas[
        df_lineas["linea"].str.contains(
            "Selo:",
            case=False,
            na=False,
            regex=False
        )
    ]

    if not coincidencias_sello.empty:
        orden_sello = coincidencias_sello.iloc[0]["orden"]

        datos["fecha_difusion_plataforma_web"] = df_lineas.loc[
            df_lineas["orden"] == orden_sello - 1,
            "linea"
        ].iloc[0]

        datos["sello_web"] = df_lineas.loc[
            df_lineas["orden"] == orden_sello + 1,
            "linea"
        ].iloc[0]

    # CPV
    coincidencias_cpv = df_lineas[
        df_lineas["linea"].str.contains(
            "CPV - Vocabulario común de contratos públicos",
            case=False,
            na=False,
            regex=False
        )
    ]

    if not coincidencias_cpv.empty:
        inicio_cpv = coincidencias_cpv.iloc[0]["orden"]

        datos["cpv_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_cpv + 4,
            "linea"
        ].iloc[0]

        datos["cpv_lote_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_cpv + 5,
            "linea"
        ].iloc[0]

        datos["cpv_fecha_difusion_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_cpv + 6,
            "linea"
        ].iloc[0]

    # NUT
    coincidencias_nut = df_lineas[
        df_lineas["linea"].str.contains(
            "NUT - Nomenclatura",
            case=False,
            na=False,
            regex=False
        )
    ]

    if not coincidencias_nut.empty:
        inicio_nut = coincidencias_nut.iloc[0]["orden"]

        datos["nut_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_nut + 4,
            "linea"
        ].iloc[0]

        datos["nut_lote_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_nut + 5,
            "linea"
        ].iloc[0]

        datos["nut_fecha_difusion_web"] = df_lineas.loc[
            df_lineas["orden"] == inicio_nut + 6,
            "linea"
        ].iloc[0]

    return datos

In [23]:
# ===============================
# 18. Aplicar publicación, CPV y NUT Galicia
# ===============================

datos_publicacion_galicia = extraer_publicacion_cpv_nut_galicia(
    df_lineas_galicia
)

for columna, valor in datos_publicacion_galicia.items():
    info_galicia[columna] = valor

# Campos no confiables todavía porque son checkboxes
info_galicia["contrato_mixto_web"] = None
info_galicia["subasta_electronica_web"] = None

info_galicia.T

,0
licitacion_id,3b417cfd32be2e0f
detail_url,https://www.contratosdegalicia.gal/licitacion?...
estado_procedimiento_web,Adxudicado
organo_contratacion_web,Consellería de Sanidade - SERGAS
objeto_web,PSM001937990 708506 biombo pregable 7 corpos...
tipo_tramitacion_web,De Emerxencia
tipo_procedimiento_web,Procedemento de emerxencia
tipo_contrato_web,Subministracións
presupuesto_base_web,"7.170,00"
num_lotes_web,_


In [24]:
# ===============================
# 19. Seleccionar licitación Madrid
# ===============================

muestra_madrid = muestra[muestra["portal"] == "madrid"].copy()

licitacion_prueba_madrid = muestra_madrid.iloc[0]

licitacion_id_madrid = licitacion_prueba_madrid["licitacion_id"]
url_madrid = licitacion_prueba_madrid["detail_url"]

print("Licitación:", licitacion_id_madrid)
print("URL:", url_madrid)

Licitación: 710a5a77b1e2940a
URL: https://contratos-publicos.comunidad.madrid/contrato-publico/servicio-tecnico-asistencial-monitorizacion-pacientes-patologia-epoc-yo-0


In [25]:
# ===============================
# 20. Extraer texto visible Madrid
# ===============================

respuesta = requests.get(url_madrid, timeout=20)
respuesta.raise_for_status()

soup_madrid = BeautifulSoup(respuesta.text, "html.parser")

texto_madrid = soup_madrid.get_text("\n", strip=True)

lineas_madrid = texto_madrid.split("\n")

df_lineas_madrid = pd.DataFrame({
    "orden": range(len(lineas_madrid)),
    "linea": lineas_madrid
})

df_lineas_madrid.head(100)

,orden,linea
0,0,Servicio Técnico-Asistencial de Monitorización...
1,1,X
2,2,Fecha y hora oficial de la Comunidad de Madrid
3,3,Fecha:
4,4,Hora:
...,...,...
95,95,Presupuesto base licitación. Importe total
96,96,"254.000,00 euros"
97,97,Duración del contrato
98,98,12 meses


In [26]:
# ===============================
# 21. Buscar secciones Madrid
# ===============================

secciones_madrid = [
    "Datos del expediente",
    "Preparación del contrato",
    "Convocatoria",
    "Pliegos de condiciones",
    "Documentación complementaria",
    "Información adicional y puntos de contacto",
    "Tablón de anuncios electrónico",
    "Resultados de la licitación"
]

df_lineas_madrid[
    df_lineas_madrid["linea"].str.contains(
        "|".join(secciones_madrid),
        case=False,
        na=False,
        regex=True
    )
]

,orden,linea
18,18,Datos del expediente
19,19,Preparación del contrato
20,20,Convocatoria
21,21,Pliegos de condiciones
22,22,Documentación complementaria
23,23,Información adicional y puntos de contacto
24,24,Tablón de anuncios electrónico
25,25,Resultados de la licitación
41,41,Datos del expediente
44,44,Convocatoria anunciada a licitación


In [27]:
# ===============================
# 22. Extraer Datos del expediente Madrid
# ===============================

info_madrid = {
    "licitacion_id": licitacion_id_madrid,
    "detail_url": url_madrid,

    "titulo_web": df_lineas_madrid.loc[
        df_lineas_madrid["orden"] == 0,
        "linea"
    ].iloc[0],

    "tipo_publicacion_web": valor_siguiente(
        df_lineas_madrid,
        "Tipo de publicación"
    ),

    "situacion_web": valor_siguiente(
        df_lineas_madrid,
        "Situación"
    ),

    "tipo_resolucion_web": valor_siguiente(
        df_lineas_madrid,
        "Tipo de resolución"
    ),

    "numero_expediente_web": valor_siguiente(
        df_lineas_madrid,
        "Número de expediente"
    ),

    "referencia_web": valor_siguiente(
        df_lineas_madrid,
        "Referencia"
    ),

    "identificador_ted_web": valor_siguiente(
        df_lineas_madrid,
        "Identificador del expediente en TED"
    ),

    "codigo_dir3_web": valor_siguiente(
        df_lineas_madrid,
        "Código de la entidad adjudicadora"
    ),

    "entidad_adjudicadora_web": valor_siguiente(
        df_lineas_madrid,
        "Entidad adjudicadora"
    ),

    "objeto_contrato_web": valor_siguiente(
        df_lineas_madrid,
        "Objeto del contrato"
    ),

    "tipo_contrato_web": valor_siguiente(
        df_lineas_madrid,
        "Tipo de contrato"
    ),

    "contrato_mixto_web": valor_siguiente(
        df_lineas_madrid,
        "Contrato mixto"
    ),

    "codigo_cpv_web": valor_siguiente(
        df_lineas_madrid,
        "Código CPV"
    ),

    "legislacion_nacional_web": valor_siguiente(
        df_lineas_madrid,
        "Legislación nacional aplicable"
    ),

    "sujeto_regulacion_armonizada_web": valor_siguiente(
        df_lineas_madrid,
        "Sujeto a regulación armonizada"
    ),

    "sistema_contratacion_web": valor_siguiente(
        df_lineas_madrid,
        "Sistema de contratación"
    ),

    "codigo_nuts_web": valor_siguiente(
        df_lineas_madrid,
        "Código NUTS"
    ),

    "compra_publica_innovacion_web": valor_siguiente(
        df_lineas_madrid,
        "Compra pública de innovación"
    ),

    "financiacion_ue_web": valor_siguiente(
        df_lineas_madrid,
        "Financiación de la Unión Europea"
    ),

    "procedimiento_adjudicacion_web": valor_siguiente(
        df_lineas_madrid,
        "Procedimiento de adjudicación"
    ),

    "tipo_tramitacion_web": valor_siguiente(
        df_lineas_madrid,
        "Tipo de tramitación"
    ),

    "metodo_presentacion_ofertas_web": valor_siguiente(
        df_lineas_madrid,
        "Método de presentación de ofertas"
    ),

    "subasta_electronica_web": valor_siguiente(
        df_lineas_madrid,
        "Subasta electrónica"
    ),

    "valor_estimado_sin_impuestos_web": valor_siguiente(
        df_lineas_madrid,
        "Valor estimado sin impuestos"
    ),

    "presupuesto_base_sin_impuestos_web": valor_siguiente(
        df_lineas_madrid,
        "Presupuesto base licitación sin impuestos"
    ),

    "presupuesto_base_total_web": valor_siguiente(
        df_lineas_madrid,
        "Presupuesto base licitación. Importe total"
    ),

    "duracion_contrato_web": valor_siguiente(
        df_lineas_madrid,
        "Duración del contrato"
    ),

    "fecha_limite_presentacion_web": valor_siguiente(
        df_lineas_madrid,
        "Fecha y hora límite de presentación de ofertas o solicitudes de participación"
    )
}

info_madrid = pd.DataFrame([info_madrid])

info_madrid.T

,0
licitacion_id,710a5a77b1e2940a
detail_url,https://contratos-publicos.comunidad.madrid/co...
titulo_web,Servicio Técnico-Asistencial de Monitorización...
tipo_publicacion_web,Convocatoria anunciada a licitación
situacion_web,Resuelta
tipo_resolucion_web,Desistimiento
numero_expediente_web,A/SER-023467/2024
referencia_web,C5816
identificador_ted_web,37798af1-4c7f-41e5-b7af-56c4a21fd653
codigo_dir3_web,A13013774


In [28]:
# ===============================
# 23. Extraer enlaces Madrid
# ===============================

enlaces_madrid = []

for i, enlace in enumerate(soup_madrid.find_all("a", href=True)):
    texto = enlace.get_text(" ", strip=True)
    href = enlace.get("href")
    url_completa = urljoin(url_madrid, href)

    enlaces_madrid.append(
        {
            "orden": i,
            "texto_enlace": texto,
            "href": href,
            "url_completa": url_completa
        }
    )

df_enlaces_madrid = pd.DataFrame(enlaces_madrid)

df_enlaces_madrid.head(50)

,orden,texto_enlace,href,url_completa
0,0,Pasar al contenido principal,#main-content,https://contratos-publicos.comunidad.madrid/co...
1,1,,/,https://contratos-publicos.comunidad.madrid/
2,2,,http://www.comunidad.madrid,http://www.comunidad.madrid
3,3,,http://www.comunidad.madrid,http://www.comunidad.madrid
4,4,Perfil de contratante,/perfil-contratante,https://contratos-publicos.comunidad.madrid/pe...
5,5,Contratación centralizada,/contratacion-centralizada,https://contratos-publicos.comunidad.madrid/co...
6,6,Servicios y consultas,/servicios-consultas,https://contratos-publicos.comunidad.madrid/se...
7,7,Información general,/informacion-general,https://contratos-publicos.comunidad.madrid/in...
8,8,Avisos y novedades,/avisos,https://contratos-publicos.comunidad.madrid/av...
9,9,2026-3-12,/contrato-publico,https://contratos-publicos.comunidad.madrid/co...


In [29]:
# ===============================
# 24. Filtrar documentos Madrid
# ===============================

palabras_docs_madrid = [
    "memoria",
    "informe",
    "iniciación",
    "aprobación",
    "anuncio",
    "pliego",
    "condiciones",
    "descargar",
    "documento",
    "pdf",
    "contrato"
]

df_docs_madrid = df_enlaces_madrid[
    df_enlaces_madrid["texto_enlace"].str.contains(
        "|".join(palabras_docs_madrid),
        case=False,
        na=False,
        regex=True
    )
].copy()

df_docs_madrid

,orden,texto_enlace,href,url_completa
11,11,Preparación del contrato,#pcon-prep-del-contrato,https://contratos-publicos.comunidad.madrid/co...
13,13,Pliegos de condiciones,#pcon-pliego-de-condiciones,https://contratos-publicos.comunidad.madrid/co...
16,16,Tablón de anuncios electrónico,#pcon-tablon,https://contratos-publicos.comunidad.madrid/co...
20,20,PDF,/contrato-publico/print/pdf/node/281213,https://contratos-publicos.comunidad.madrid/co...
22,22,Descargar,/medias/memoriadenecesidades20240701signedpdf/...,https://contratos-publicos.comunidad.madrid/me...
23,23,Descargar,/medias/memoriaeconomica20240701v5signedpdf/do...,https://contratos-publicos.comunidad.madrid/me...
24,24,Descargar,/medias/ser-023467-2024resolucioniniciofpdf/do...,https://contratos-publicos.comunidad.madrid/me...
25,25,Descargar,/medias/ser-023467-2024resolaprobacionpliegosf...,https://contratos-publicos.comunidad.madrid/me...
26,26,Descargar,/medias/memoriaeconomica20240701v5signed0pdf/d...,https://contratos-publicos.comunidad.madrid/me...
28,28,Descargar todos los archivos,/generate-zip/281213/group_pcon_prep_del_contrato,https://contratos-publicos.comunidad.madrid/ge...


In [30]:
# ===============================
# 25. Seleccionar documento de prueba
# ===============================

doc_prueba = df_docs_madrid.iloc[0]

url_pdf = doc_prueba["url_completa"]
nombre_doc = doc_prueba["texto_enlace"]

print(nombre_doc)
print(url_pdf)

Preparación del contrato
https://contratos-publicos.comunidad.madrid/contrato-publico/servicio-tecnico-asistencial-monitorizacion-pacientes-patologia-epoc-yo-0#pcon-prep-del-contrato


In [41]:
# ===============================
# Descargar documentos detectando formato real
# ===============================

from pathlib import Path
from time import sleep
import requests
import pandas as pd
import re


def limpiar_nombre_archivo(texto):
    texto = str(texto)
    texto = re.sub(r"[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ_ -]", "", texto)
    texto = texto.strip().replace(" ", "_")
    return texto[:80]


def detectar_extension(contenido, content_type, url):
    """
    Detecta la extensión real del archivo descargado.
    """
    content_type = str(content_type).lower()
    url = str(url).lower()

    if contenido.startswith(b"%PDF") or "application/pdf" in content_type:
        return ".pdf", "PDF"

    if contenido.startswith(b"PK"):
        return ".zip", "ZIP_O_OFFICE"

    if contenido.startswith(b"<?xml") or "xml" in content_type:
        return ".xml", "XML"

    if contenido.startswith(b"<!DOC") or "text/html" in content_type:
        return ".html", "HTML"

    if ".pdf" in url:
        return ".pdf", "PDF_POSIBLE"

    return ".bin", "DESCONOCIDO"


ruta_bronze = Path(
    r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos"
)

ruta_licitacion = ruta_bronze / str(licitacion_id_madrid)
ruta_licitacion.mkdir(parents=True, exist_ok=True)

descargas = []

for i, fila in df_docs_madrid.reset_index(drop=True).iterrows():

    nombre_doc = fila["texto_enlace"]
    url_doc = fila["url_completa"]

    try:
        respuesta = requests.get(url_doc, timeout=30)
        respuesta.raise_for_status()

        contenido = respuesta.content
        content_type = respuesta.headers.get("Content-Type", "")

        extension, tipo_archivo = detectar_extension(
            contenido,
            content_type,
            url_doc
        )

        nombre_archivo = (
            f"{i + 1:02d}_"
            f"{limpiar_nombre_archivo(nombre_doc)}"
            f"{extension}"
        )

        ruta_archivo = ruta_licitacion / nombre_archivo

        with open(ruta_archivo, "wb") as archivo:
            archivo.write(contenido)

        descargas.append(
            {
                "licitacion_id": licitacion_id_madrid,
                "nombre_documento": nombre_doc,
                "url_documento": url_doc,
                "content_type": content_type,
                "tipo_archivo_detectado": tipo_archivo,
                "ruta_archivo": str(ruta_archivo),
                "estado_descarga": "OK",
                "error_descarga": None
            }
        )

        print("Guardado:", tipo_archivo, ruta_archivo)

    except Exception as error:
        descargas.append(
            {
                "licitacion_id": licitacion_id_madrid,
                "nombre_documento": nombre_doc,
                "url_documento": url_doc,
                "content_type": None,
                "tipo_archivo_detectado": None,
                "ruta_archivo": None,
                "estado_descarga": "ERROR",
                "error_descarga": str(error)
            }
        )

        print("Error:", nombre_doc, error)

    sleep(1)


df_descargas_documentos = pd.DataFrame(descargas)

df_descargas_documentos

Guardado: HTML C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\01_Preparación_del_contrato.html
Guardado: HTML C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\02_Pliegos_de_condiciones.html
Guardado: HTML C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\03_Tablón_de_anuncios_electrónico.html
Guardado: PDF C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\04_PDF.pdf
Guardado: PDF C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\05_Descargar.pdf
Guardado: PDF C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\06_Descargar.pdf
Guardado: PDF C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\07_Descargar.pdf
Guardado: PDF C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bro

,licitacion_id,nombre_documento,url_documento,content_type,tipo_archivo_detectado,ruta_archivo,estado_descarga,error_descarga
0,710a5a77b1e2940a,Preparación del contrato,https://contratos-publicos.comunidad.madrid/co...,text/html; charset=UTF-8,HTML,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
1,710a5a77b1e2940a,Pliegos de condiciones,https://contratos-publicos.comunidad.madrid/co...,text/html; charset=UTF-8,HTML,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
2,710a5a77b1e2940a,Tablón de anuncios electrónico,https://contratos-publicos.comunidad.madrid/co...,text/html; charset=UTF-8,HTML,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
3,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
4,710a5a77b1e2940a,Descargar,https://contratos-publicos.comunidad.madrid/me...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
5,710a5a77b1e2940a,Descargar,https://contratos-publicos.comunidad.madrid/me...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
6,710a5a77b1e2940a,Descargar,https://contratos-publicos.comunidad.madrid/me...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
7,710a5a77b1e2940a,Descargar,https://contratos-publicos.comunidad.madrid/me...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
8,710a5a77b1e2940a,Descargar,https://contratos-publicos.comunidad.madrid/me...,application/pdf,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None
9,710a5a77b1e2940a,Descargar todos los archivos,https://contratos-publicos.comunidad.madrid/ge...,application/zip,ZIP_O_OFFICE,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,OK,None


In [42]:
# ===============================
# Guardar metadata corregida en Bronze
# ===============================

ruta_metadata = ruta_licitacion / "metadata_documentos.csv"

df_descargas_documentos.to_csv(
    ruta_metadata,
    index=False,
    encoding="utf-8-sig"
)

print("Metadata guardada en:", ruta_metadata)

Metadata guardada en: C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\bronze\documentos\710a5a77b1e2940a\metadata_documentos.csv


In [44]:
# ===============================
# Filtrar solo PDFs reales
# ===============================

df_metadata_pdf = df_descargas_documentos[
    (df_descargas_documentos["estado_descarga"] == "OK")
    & (df_descargas_documentos["tipo_archivo_detectado"].isin(["PDF", "PDF_POSIBLE"]))
].copy()

df_metadata_pdf[
    [
        "nombre_documento",
        "tipo_archivo_detectado",
        "ruta_archivo"
    ]
]

,nombre_documento,tipo_archivo_detectado,ruta_archivo
3,PDF,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
4,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
5,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
6,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
7,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
8,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
10,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
12,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
13,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
14,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...


In [45]:
# ===============================
# Filtrar solo PDFs reales
# ===============================

df_metadata_pdf = df_descargas_documentos[
    (df_descargas_documentos["estado_descarga"] == "OK")
    & (df_descargas_documentos["tipo_archivo_detectado"].isin(["PDF", "PDF_POSIBLE"]))
].copy()

df_metadata_pdf[
    [
        "nombre_documento",
        "tipo_archivo_detectado",
        "ruta_archivo"
    ]
]

,nombre_documento,tipo_archivo_detectado,ruta_archivo
3,PDF,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
4,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
5,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
6,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
7,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
8,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
10,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
12,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
13,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...
14,Descargar,PDF,C:\Users\Usuario\TFM\TFM_scraping_contratacion...


In [46]:
# ===============================
# Extraer texto solo de PDFs
# ===============================

from pypdf import PdfReader


def extraer_texto_pdf(ruta_pdf):
    try:
        reader = PdfReader(ruta_pdf)

        textos_paginas = []

        for pagina in reader.pages:
            texto_pagina = pagina.extract_text()

            if texto_pagina:
                textos_paginas.append(texto_pagina)

        return {
            "texto_extraido": "\n".join(textos_paginas),
            "numero_paginas": len(reader.pages),
            "estado_lectura": "OK",
            "error_lectura": None
        }

    except Exception as error:
        return {
            "texto_extraido": None,
            "numero_paginas": None,
            "estado_lectura": "ERROR",
            "error_lectura": str(error)
        }


resultados_texto = []

for _, fila in df_metadata_pdf.iterrows():

    resultado = extraer_texto_pdf(fila["ruta_archivo"])

    resultados_texto.append(
        {
            "licitacion_id": fila["licitacion_id"],
            "nombre_documento": fila["nombre_documento"],
            "url_documento": fila["url_documento"],
            "ruta_archivo_bronze": fila["ruta_archivo"],
            "tipo_archivo_detectado": fila["tipo_archivo_detectado"],
            "texto_extraido": resultado["texto_extraido"],
            "numero_paginas": resultado["numero_paginas"],
            "estado_lectura": resultado["estado_lectura"],
            "error_lectura": resultado["error_lectura"]
        }
    )


df_textos_pdf = pd.DataFrame(resultados_texto)

df_textos_pdf[
    [
        "nombre_documento",
        "numero_paginas",
        "estado_lectura",
        "error_lectura"
    ]
]

,nombre_documento,numero_paginas,estado_lectura,error_lectura
0,PDF,6,OK,None
1,Descargar,2,OK,None
2,Descargar,1,OK,None
3,Descargar,2,OK,None
4,Descargar,1,OK,None
5,Descargar,6,OK,None
6,Descargar,2,OK,None
7,Descargar,76,OK,None
8,Descargar,56,OK,None
9,Descargar,1,OK,None


In [47]:
# ===============================
# Guardar textos PDF en capa Silver
# ===============================

from pathlib import Path

ruta_silver_documentos = Path(
    r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\documentos_texto"
)

ruta_silver_licitacion = ruta_silver_documentos / str(licitacion_id_madrid)

ruta_silver_licitacion.mkdir(parents=True, exist_ok=True)

ruta_textos_pdf = ruta_silver_licitacion / "textos_documentos_pdf.parquet"

df_textos_pdf.to_parquet(
    ruta_textos_pdf,
    index=False
)

print("Textos PDF guardados en:", ruta_textos_pdf)

Textos PDF guardados en: C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\documentos_texto\710a5a77b1e2940a\textos_documentos_pdf.parquet


In [48]:
# ===============================
# Validar archivo Silver
# ===============================

df_validacion_textos = pd.read_parquet(ruta_textos_pdf)

df_validacion_textos[
    [
        "licitacion_id",
        "nombre_documento",
        "numero_paginas",
        "estado_lectura",
        "error_lectura"
    ]
]

,licitacion_id,nombre_documento,numero_paginas,estado_lectura,error_lectura
0,710a5a77b1e2940a,PDF,6,OK,None
1,710a5a77b1e2940a,Descargar,2,OK,None
2,710a5a77b1e2940a,Descargar,1,OK,None
3,710a5a77b1e2940a,Descargar,2,OK,None
4,710a5a77b1e2940a,Descargar,1,OK,None
5,710a5a77b1e2940a,Descargar,6,OK,None
6,710a5a77b1e2940a,Descargar,2,OK,None
7,710a5a77b1e2940a,Descargar,76,OK,None
8,710a5a77b1e2940a,Descargar,56,OK,None
9,710a5a77b1e2940a,Descargar,1,OK,None


In [49]:
# ===============================
# Buscar términos relevantes por documento
# ===============================

terminos_relevantes = [
    "objeto",
    "solvencia",
    "criterios de adjudicación",
    "prescripciones técnicas",
    "personal",
    "medios personales",
    "medios materiales",
    "experiencia",
    "certificación",
    "obligaciones",
    "plazo",
    "duración",
    "lugar de prestación",
    "presupuesto"
]

resultados_terminos = []

for _, fila in df_textos_pdf.iterrows():
    texto = str(fila["texto_extraido"]).lower()

    for termino in terminos_relevantes:
        resultados_terminos.append(
            {
                "licitacion_id": fila["licitacion_id"],
                "nombre_documento": fila["nombre_documento"],
                "numero_paginas": fila["numero_paginas"],
                "termino": termino,
                "encontrado": termino.lower() in texto
            }
        )

df_terminos_pdf = pd.DataFrame(resultados_terminos)

df_terminos_pdf.groupby("termino")["encontrado"].sum().sort_values(
    ascending=False
)

termino
objeto                       9
prescripciones técnicas      8
presupuesto                  7
plazo                        7
duración                     5
criterios de adjudicación    4
solvencia                    4
personal                     3
experiencia                  3
certificación                2
obligaciones                 2
medios materiales            1
medios personales            1
lugar de prestación          0
Name: encontrado, dtype: int64

In [50]:
# ===============================
# 1. Función para extraer fragmentos de evidencia
# ===============================

def extraer_fragmentos(texto, termino, ventana=600):
    """
    Extrae fragmentos alrededor de todas las apariciones de un término.
    """
    texto = str(texto)
    texto_lower = texto.lower()
    termino_lower = termino.lower()

    fragmentos = []
    inicio_busqueda = 0

    while True:
        posicion = texto_lower.find(termino_lower, inicio_busqueda)

        if posicion == -1:
            break

        inicio = max(0, posicion - ventana)
        fin = min(len(texto), posicion + len(termino) + ventana)

        fragmento = texto[inicio:fin].replace("\n", " ").strip()

        fragmentos.append(fragmento)

        inicio_busqueda = posicion + len(termino)

    return fragmentos

In [51]:
# ===============================
# 2. Crear tabla de evidencias documentales
# ===============================

terminos_relevantes = [
    "objeto",
    "solvencia",
    "criterios de adjudicación",
    "prescripciones técnicas",
    "personal",
    "medios personales",
    "medios materiales",
    "experiencia",
    "certificación",
    "obligaciones",
    "plazo",
    "duración",
    "lugar de prestación",
    "presupuesto"
]

evidencias = []

for _, fila in df_textos_pdf.iterrows():

    texto = fila["texto_extraido"]

    for termino in terminos_relevantes:

        fragmentos = extraer_fragmentos(
            texto=texto,
            termino=termino,
            ventana=600
        )

        for i, fragmento in enumerate(fragmentos, start=1):

            evidencias.append(
                {
                    "licitacion_id": fila["licitacion_id"],
                    "nombre_documento": fila["nombre_documento"],
                    "url_documento": fila["url_documento"],
                    "ruta_archivo_bronze": fila["ruta_archivo_bronze"],
                    "numero_paginas": fila["numero_paginas"],
                    "termino": termino,
                    "numero_fragmento": i,
                    "fragmento_evidencia": fragmento
                }
            )

df_evidencias_pdf = pd.DataFrame(evidencias)

df_evidencias_pdf.head()

,licitacion_id,nombre_documento,url_documento,ruta_archivo_bronze,numero_paginas,termino,numero_fragmento,fragmento_evidencia
0,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,6,objeto,1,ión de ofertas o solicitudes de participación:...
1,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,6,prescripciones técnicas,1,icitación (Publicado el 8 de julio del 2024 14...
2,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,6,plazo,1,En plazo Pendiente de adjudicación Adjudicada ...
3,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,6,duración,1,os Contrato mixto No Código CPV 85100000- 0...
4,710a5a77b1e2940a,PDF,https://contratos-publicos.comunidad.madrid/co...,C:\Users\Usuario\TFM\TFM_scraping_contratacion...,6,presupuesto,1,o Insuficiencia Cardiaca y/o Paciente Crónico ...


In [52]:
# ===============================
# 3. Resumen de evidencias
# ===============================

df_evidencias_pdf.groupby("termino").size().sort_values(ascending=False)

termino
personal                     113
plazo                        109
objeto                        72
obligaciones                  50
solvencia                     38
duración                      33
prescripciones técnicas       27
presupuesto                   22
criterios de adjudicación     15
certificación                 13
experiencia                    9
medios personales              2
medios materiales              1
dtype: int64

In [53]:
# ===============================
# 4. Ver evidencias de solvencia
# ===============================

pd.set_option("display.max_colwidth", 1000)

df_evidencias_pdf[
    df_evidencias_pdf["termino"] == "solvencia"
][
    [
        "nombre_documento",
        "termino",
        "fragmento_evidencia"
    ]
].head(5)

,nombre_documento,termino,fragmento_evidencia
16,Descargar,solvencia,"cesidad de tramitar este expediente de contratación con las siguientes características: 1. Objeto del contrato. (art. 99 Ley 9/2017 LCSP). El objeto del presente contrato es el: “SERVICIO TÉCNICO ASISTENCIAL DE MONITORIZACIÓN DE PACIENTES CON PATOLOGÍA EPOC Y/O INSUFICIENCIA CARDIACA Y/O PACIENTE CRÓNICO DEL HOSPITAL GENERAL UNIVERSITARIO GREGORIO MARAÑÓN "" División en lotes. (art. 99 LCSP). Dada la naturaleza del objeto del contrato, este NO está dividido en lotes. 2. Valor estimado del contrato. (artículo 101 LCSP). El valor estimado del contrato es de: 461.818,19 euros. 3. Solvencias exigidas. Los criterios de solvencia requeridos están vinculados al objeto del contrato, son proporcionales al mismo y tiene como finalidad garantizar que el adjudicatario dispone de los medios adecuados para la correcta ejecución del contrato. Para la acreditación de la solvencia económica y financiera, de los previstos en el artículo 87 de la LCSP, se seleccionan los siguient..."
17,Descargar,solvencia,"contratación con las siguientes características: 1. Objeto del contrato. (art. 99 Ley 9/2017 LCSP). El objeto del presente contrato es el: “SERVICIO TÉCNICO ASISTENCIAL DE MONITORIZACIÓN DE PACIENTES CON PATOLOGÍA EPOC Y/O INSUFICIENCIA CARDIACA Y/O PACIENTE CRÓNICO DEL HOSPITAL GENERAL UNIVERSITARIO GREGORIO MARAÑÓN "" División en lotes. (art. 99 LCSP). Dada la naturaleza del objeto del contrato, este NO está dividido en lotes. 2. Valor estimado del contrato. (artículo 101 LCSP). El valor estimado del contrato es de: 461.818,19 euros. 3. Solvencias exigidas. Los criterios de solvencia requeridos están vinculados al objeto del contrato, son proporcionales al mismo y tiene como finalidad garantizar que el adjudicatario dispone de los medios adecuados para la correcta ejecución del contrato. Para la acreditación de la solvencia económica y financiera, de los previstos en el artículo 87 de la LCSP, se seleccionan los siguientes medios: párrafo 3 letra a) o b) y c)..."
18,Descargar,solvencia,"IACA Y/O PACIENTE CRÓNICO DEL HOSPITAL GENERAL UNIVERSITARIO GREGORIO MARAÑÓN "" División en lotes. (art. 99 LCSP). Dada la naturaleza del objeto del contrato, este NO está dividido en lotes. 2. Valor estimado del contrato. (artículo 101 LCSP). El valor estimado del contrato es de: 461.818,19 euros. 3. Solvencias exigidas. Los criterios de solvencia requeridos están vinculados al objeto del contrato, son proporcionales al mismo y tiene como finalidad garantizar que el adjudicatario dispone de los medios adecuados para la correcta ejecución del contrato. Para la acreditación de la solvencia económica y financiera, de los previstos en el artículo 87 de la LCSP, se seleccionan los siguientes medios: párrafo 3 letra a) o b) y c) si la antigüedad de la empresa es inferior a tres años Para la acreditación de la solvencia técnica o profesional, de los medios señalados en el artículo 90 de la LCSP, el apartado 1, letra a) y b). 4. Procedimiento y criterios de adjudicaci..."
19,Descargar,solvencia,"rtículo 101 LCSP). El valor estimado del contrato es de: 461.818,19 euros. 3. Solvencias exigidas. Los criterios de solvencia requeridos están vinculados al objeto del contrato, son proporcionales al mismo y tiene como finalidad garantizar que el adjudicatario dispone de los medios adecuados para la correcta ejecución del contrato. Para la acreditación de la solvencia económica y financiera, de los previstos en el artículo 87 de la LCSP, se seleccionan los siguientes medios: párrafo 3 letra a) o b) y c) si la antigüedad de la empresa es inferior a tres años Para la acreditación de la solvencia técnica o profesional, de los medios señalados en el artículo 90 de la LCSP, el apartado 1, letra a) y b). 4. Procedimiento y criterios de adjudicación. Considerando el importe del valor estimado y de conformidad con el artículo 131 de la LCSP, el procedimiento de adjudicación de este contrato es abierto con plural